In [1]:
import numpy as np
from toad import TOAD
from toad.shifts_detection.methods import ASDETECT
from sklearn.cluster import HDBSCAN
from toad.visualisation.toad_plotter import PlotConfig
from ponds import PONDS

In [2]:
pd = PONDS(lat_size=10,
           lon_size=10,
           time_size=100,
            n_shifts=3)

#num = pd.ts_param["sig_num"].values
#times = pd.ts_param["sig_tim"].values
#steepness = pd.ts_param["sig_ste"].values

#print(num)
#print(times)
#print(steepness)

pd.generate_shifts()

<xarray.Dataset> Size: 4kB
Dimensions:  (lat: 10, lon: 10)
Coordinates:
  * lat      (lat) float64 80B -90.0 -70.0 -50.0 -30.0 ... 30.0 50.0 70.0 90.0
  * lon      (lon) float64 80B -180.0 -140.0 -100.0 -60.0 ... 100.0 140.0 180.0
Data variables:
    sig_num  (lat, lon) int64 800B 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 1 0 0 0
    sig_tim  (lat, lon) object 800B None None None None ... None None None
    sig_tsg  (lat, lon) object 800B None None None None ... None None None
    sig_ste  (lat, lon) object 800B None None None None ... None None None
    sig_mag  (lat, lon) object 800B None None None None ... None None None


## Sandbox

In [4]:
import xarray as xr

data = xr.DataArray(np.random.randn(2, 3),
                    dims=("x", "y"),
                    coords={"x": [10, 20]},
                    )
data

# like in pandas, values is a numpy array that you can modify in-place
#data.values
#data.dims
#data.coords
# you can use this dictionary to store arbitrary metadata
#data.attrs

<xarray.DataArray (x: 2, y: 3)> Size: 48B
array([[ 0.51434587,  0.47025281, -0.83119695],
       [-0.91006472, -0.26694958, -1.04619993]])
Coordinates:
  * x        (x) int64 16B 10 20
Dimensions without coordinates: y

In [17]:
import numpy as np
import xarray as xr

lats = np.arange(2)
lons = np.arange(2)

ts = [3,4]

# Create the Dataset
ts_param = xr.Dataset(coords={"lat": lats, "lon": lons})

# Boolean mask
mask = np.array([[True, False],
                 [False, True]])

# Create empty object array
ragged_data = np.empty(mask.shape, dtype=object)

for t in ts:
    for i in range(len(lats)):
        for j in range(len(lons)):
            if mask[i, j]:
                if ragged_data[i, j] is None:
                    ragged_data[i, j] = [t]
                else:
                    ragged_data[i, j].append(t)  # Fill with a list containing 1

# Fill: [1] if True, [] if False
#ragged_data[:] = np.where(mask, object(), object())  # allocate space

# Use broadcasting to fill
#ragged_data[mask] = [ 1 for _ in range(mask.sum()) ]
#ragged_data[~mask] = [ 0 for _ in range((~mask).sum()) ]

# Assign to Dataset
ts_param["extra_data"] = (("lat", "lon"), ragged_data)

print(type(ts_param["extra_data"]))

print(ts_param)


<class 'xarray.core.dataarray.DataArray'>
<xarray.Dataset> Size: 64B
Dimensions:     (lat: 2, lon: 2)
Coordinates:
  * lat         (lat) int64 16B 0 1
  * lon         (lon) int64 16B 0 1
Data variables:
    extra_data  (lat, lon) object 32B [3, 4] None None [3, 4]


In [6]:
lats = np.arange(2)
lons = np.arange(2)

ts_param = xr.Dataset(
            coords={
                "lat": lats,
                "lon": lons,
            }
        )

ts_param["hs_num"] = (("lat", "lon"), np.array([[0,1],
                                                [0,0]]))
hs_tim = np.empty((2, 2), dtype=object)
hs_tim[0, 1] = [3]
hs_tim[0, 0] = []

ts_param["hs_tim"] = (("lat", "lon"), hs_tim)
ts_param

<xarray.Dataset> Size: 96B
Dimensions:  (lat: 2, lon: 2)
Coordinates:
  * lat      (lat) int64 16B 0 1
  * lon      (lon) int64 16B 0 1
Data variables:
    hs_num   (lat, lon) int64 32B 0 1 0 0
    hs_tim   (lat, lon) object 32B [] [3] None None

In [ ]:
lats = np.arange(30)
lons = np.arange(20)
blob_normalized = np.random.rand(len(lats), len(lons) * 3)  # Simulating a blob with periodic longitudes
#print(blob_normalized)
                    #np.array([[0.1, 0.4, 0.3, 0.2, 0.6, 0.1,],
                  #          [0.3, 0.2, 0.3, 0.2, 0.6, 0.1,]])


bc_old = np.zeros((len(lats),len(lons)))
for i in range(len(lats)):
    #print(f"Processing latitude index i: {i}")
    for j in range(len(lons)):
        #print(f"Processing longitude index j: {j}")
        # Sum contributions from periodic longitudes - lon grid is repeated 3 times
        #print(i*len(lons) + j, '->', i, ',', j)
        bc_old[i,j] = np.sum(
            [blob_normalized[i, j + k * len(lons)] for k in range(3)]
        )
        #print(i, ',', j, ':',blob_contribution)



bc = np.sum(blob_normalized.reshape(len(lats),3,len(lons)).transpose(1, 0, 2), axis=0).reshape(len(lats), len(lons))

print(bc_old)
print(bc)

print(f"Arrays equal: {np.allclose(bc_old, bc)}")

[[0.66942848 1.26279605 1.39956133 1.43086755 1.6603589  1.29134888
  0.97712013 0.97773357 0.52911524 1.27285797 1.13791348 1.56119428
  2.07975328 1.40388681 1.83420324 1.34030106 1.98885013 1.18227594
  0.64999044 1.7717884 ]
 [1.23358317 1.23833457 1.27152934 1.41126427 0.54951045 0.90023329
  1.17290502 0.93342576 1.51283505 1.0724047  2.18991163 1.06053981
  2.77591158 0.81576687 0.62970357 2.28090491 1.12822094 1.0632811
  1.0667457  1.34798673]
 [1.80347754 1.66932363 2.61944493 0.87320865 2.06978382 2.42470729
  1.69238219 1.76949941 1.74680342 1.30488955 1.81444932 0.81524031
  1.37762966 1.41660287 1.83910947 1.82758795 1.76324588 1.53424463
  2.23064123 1.23942069]
 [0.9296706  1.38426458 0.84376382 1.72362902 1.13279851 1.47370052
  1.22481766 0.52533974 0.74718845 1.51905391 2.04894582 1.74583836
  0.96983513 1.80492191 0.92615981 1.11946056 1.83699077 1.67844771
  1.6721691  1.3931954 ]
 [1.08197765 1.01291062 1.29121215 2.40558992 1.2776777  1.70377252
  1.2784581  1.76

In [ ]:
bn = np.array([[1, 2, 3],
               [4, 5, 6]])
x = len(bn)
y = len(bn[0])//3

bn_new = np.array([[[1,2],
                    [7,8]],
                    [[3,4],
                     [9,10]],
                     [[5,6],
                      [11,12]]])

bn_try = bn.reshape(x,3,y).transpose(1,0,2)

print(bn_try)
bc_try = np.sum(bn_try, axis=0).reshape((x,y))
print(bc_try)

[[[1]
  [4]]

 [[2]
  [5]]

 [[3]
  [6]]]
[[ 6]
 [15]]
